# 🛒 电商用户消费行为分析与RFM客户分群系统

**项目周期**：2026.02 — 2026.04  
**数据来源**：[UCI Online Retail Dataset](https://archive.ics.uci.edu/ml/datasets/online+retail)  
**数据规模**：英国电商，54万+条交易记录  

## 项目目标

基于UCI在线零售数据集，运用RFM模型与聚类算法进行客户价值分群，为精准营销提供数据支撑。

## 技术栈

- **数据处理**：Pandas, NumPy
- **可视化**：Matplotlib, Seaborn
- **聚类算法**：Scikit-learn（K-Means）
- **评估指标**：轮廓系数（Silhouette Score）

## 1. 数据加载与探索

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据（UCI数据集为Excel格式）
df = pd.read_excel('Online Retail.xlsx')
print(f'数据集形状: {df.shape}')
print(f'总记录数: {df.shape[0]:,} 条')
print(f'字段数: {df.shape[1]} 个')
df.head()

In [ ]:
print('字段说明:')
print('='*60)
print('InvoiceNo   : 发票编号（C开头为退货订单）')
print('StockCode   : 商品编码')
print('Description : 商品描述')
print('Quantity    : 购买数量')
print('InvoiceDate : 发票日期（交易时间）')
print('UnitPrice   : 单价（英镑）')
print('CustomerID  : 客户编号')
print('Country     : 客户所在国家')
print()
print('数据基本信息:')
print('='*60)
print(df.info())

In [ ]:
print('数值字段统计:')
df.describe()

## 2. 数据清洗

In [ ]:
print('数据清洗前:')
print(f'  总记录数: {len(df):,}')
print()

# 2.1 删除CustomerID缺失的记录（无法进行客户分析）
missing_customer = df['CustomerID'].isnull().sum()
df = df.dropna(subset=['CustomerID'])
print(f'  删除CustomerID缺失: -{missing_customer:,} 条')

# 2.2 删除退货订单（InvoiceNo以C开头）
cancel_mask = df['InvoiceNo'].astype(str).str.startswith('C')
cancel_count = cancel_mask.sum()
df = df[~cancel_mask]
print(f'  删除退货订单: -{cancel_count:,} 条')

# 2.3 删除Quantity <= 0 的记录
invalid_qty = (df['Quantity'] <= 0).sum()
df = df[df['Quantity'] > 0]
print(f'  删除无效数量: -{invalid_qty:,} 条')

# 2.4 删除UnitPrice <= 0 的记录
invalid_price = (df['UnitPrice'] <= 0).sum()
df = df[df['UnitPrice'] > 0]
print(f'  删除无效价格: -{invalid_price:,} 条')

# 2.5 删除重复记录
duplicates = df.duplicated().sum()
df = df.drop_duplicates()
print(f'  删除重复记录: -{duplicates:,} 条')

print(f'\n  清洗后记录数: {len(df):,} 条')
print(f'  保留率: {len(df)/541909*100:.1f}%')

In [ ]:
# 2.6 构建交易金额
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

# 2.7 数据概览
print('清洗后数据概览:')
print('='*60)
print(f'客户数量: {df["CustomerID"].nunique():,}')
print(f'商品种类: {df["StockCode"].nunique():,}')
print(f'交易记录: {len(df):,} 条')
print(f'时间范围: {df["InvoiceDate"].min().date()} ~ {df["InvoiceDate"].max().date()}')
print(f'覆盖国家: {df["Country"].nunique()} 个')
print(f'总交易额: £{df["TotalAmount"].sum():,.2f}')

## 3. 探索性数据分析（EDA）

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3.1 月度交易量趋势
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly_orders = df.groupby('InvoiceMonth').agg(
    订单数=('InvoiceNo', 'nunique'),
    交易额=('TotalAmount', 'sum')
).reset_index()
monthly_orders['InvoiceMonth'] = monthly_orders['InvoiceMonth'].astype(str)

ax1 = axes[0, 0]
ax1.bar(range(len(monthly_orders)), monthly_orders['订单数'], color='steelblue', alpha=0.7)
ax1.set_xticks(range(len(monthly_orders)))
ax1.set_xticklabels(monthly_orders['InvoiceMonth'], rotation=45, fontsize=8)
ax1.set_title('月度订单量趋势', fontsize=13)
ax1.set_ylabel('订单数')

# 3.2 国家分布（Top 10）
country_orders = df.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False).head(10)
axes[0, 1].barh(range(len(country_orders)), country_orders.values, color='coral')
axes[0, 1].set_yticks(range(len(country_orders)))
axes[0, 1].set_yticklabels(country_orders.index)
axes[0, 1].set_title('Top 10 国家订单量', fontsize=13)
axes[0, 1].invert_yaxis()

# 3.3 客单价分布
order_amounts = df.groupby('InvoiceNo')['TotalAmount'].sum()
axes[1, 0].hist(order_amounts[order_amounts < 1000], bins=50, color='green', alpha=0.7)
axes[1, 0].axvline(order_amounts.mean(), color='red', linestyle='--', 
                   label=f'均值: £{order_amounts.mean():.0f}')
axes[1, 0].axvline(order_amounts.median(), color='blue', linestyle='--', 
                   label=f'中位数: £{order_amounts.median():.0f}')
axes[1, 0].set_title('客单价分布（<£1000）', fontsize=13)
axes[1, 0].set_xlabel('订单金额 (£)')
axes[1, 0].legend()

# 3.4 每位客户的订单数分布
customer_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()
axes[1, 1].hist(customer_orders[customer_orders <= 30], bins=30, color='purple', alpha=0.7)
axes[1, 1].axvline(customer_orders.mean(), color='red', linestyle='--', 
                   label=f'均值: {customer_orders.mean():.1f}次')
axes[1, 1].set_title('客户购买频次分布（≤30次）', fontsize=13)
axes[1, 1].set_xlabel('购买次数')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('output/01_eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 图1: 基础分析（月度趋势、国家分布、客单价、购买频次）')

## 4. RFM模型构建

In [ ]:
# 4.1 计算RFM指标
# 设定分析基准日期（数据集中最晚日期 + 1天）
snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)
print(f'分析基准日期: {snapshot_date.date()}')

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                    # Frequency
    'TotalAmount': 'sum'                                       # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print(f'\nRFM指标统计:')
print('='*60)
rfm.describe().round(2)

In [ ]:
# 4.2 RFM评分（5分制）
# Recency: 越小越好（最近购买）
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5, 4, 3, 2, 1])

# Frequency: 越大越好（购买频繁）
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# Monetary: 越大越好（消费金额高）
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# 转为整数
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)

# 计算RFM总分
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print('RFM评分分布:')
print('='*60)
print(f'R评分分布: {dict(rfm["R_Score"].value_counts().sort_index())}')
print(f'F评分分布: {dict(rfm["F_Score"].value_counts().sort_index())}')
print(f'M评分分布: {dict(rfm["M_Score"].value_counts().sort_index())}')
print(f'\nRFM总分分布:')
print(rfm['RFM_Score'].value_counts().sort_index())

In [ ]:
# 4.3 基于RFM总分的客户标签
def rfm_label(score):
    if score >= 13:
        return '高价值客户'
    elif score >= 10:
        return '潜力客户'
    elif score >= 7:
        return '一般客户'
    else:
        return '流失风险客户'

rfm['RFM_Label'] = rfm['RFM_Score'].apply(rfm_label)

label_counts = rfm['RFM_Label'].value_counts()
print('客户标签分布:')
print('='*60)
for label, count in label_counts.items():
    print(f'  {label}: {count:,} 人 ({count/len(rfm)*100:.1f}%)')

## 5. K-Means聚类

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 5.1 数据标准化
rfm_features = rfm[['Recency', 'Frequency', 'Monetary']]
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)

print('标准化前统计:')
print(rfm_features.describe().round(2))
print('\n标准化后统计:')
print(pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary']).describe().round(2))

In [ ]:
# 5.2 肘部法则确定最优簇数
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertias.append(kmeans.inertia_)
    sil = silhouette_score(rfm_scaled, kmeans.labels_)
    silhouettes.append(sil)
    print(f'K={k}: 惯性={kmeans.inertia_:,.0f}, 轮廓系数={sil:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2)
axes[0].set_title('肘部法则', fontsize=13)
axes[0].set_xlabel('簇数 K')
axes[0].set_ylabel('惯性 (Inertia)')
axes[0].axvline(x=4, color='red', linestyle='--', label='K=4')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'rs-', linewidth=2)
axes[1].set_title('轮廓系数', fontsize=13)
axes[1].set_xlabel('簇数 K')
axes[1].set_ylabel('轮廓系数')
axes[1].axvline(x=4, color='red', linestyle='--', label='K=4')
axes[1].legend()

plt.tight_layout()
plt.savefig('output/02_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 图2: 肘部法则与轮廓系数')

In [ ]:
# 5.3 使用K=4进行聚类
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# 计算每个簇的RFM均值
cluster_summary = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(2)

cluster_summary.columns = ['Recency_均值', 'Frequency_均值', 'Monetary_均值', '客户数']
print('各簇RFM均值:')
print('='*60)
print(cluster_summary)

In [ ]:
# 5.4 为每个簇打标签
def assign_cluster_label(row):
    if row['Recency'] < cluster_summary['Recency_均值'].median() and \
       row['Frequency'] > cluster_summary['Frequency_均值'].median() and \
       row['Monetary'] > cluster_summary['Monetary_均值'].median():
        return '高价值客户'
    elif row['Recency'] < cluster_summary['Recency_均值'].median() and \
         row['Frequency'] <= cluster_summary['Frequency_均值'].median():
        return '潜力客户'
    elif row['Recency'] >= cluster_summary['Recency_均值'].median() and \
         row['Frequency'] > cluster_summary['Frequency_均值'].median():
        return '一般客户'
    else:
        return '流失风险客户'

# 基于均值排序来分配标签
cluster_means = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()
# Recency越小越好，Frequency和Monetary越大越好
cluster_means['R_rank'] = cluster_means['Recency'].rank()  # 越小排名越高
cluster_means['F_rank'] = cluster_means['Frequency'].rank(ascending=False)
cluster_means['M_rank'] = cluster_means['Monetary'].rank(ascending=False)
cluster_means['total_rank'] = cluster_means['R_rank'] + cluster_means['F_rank'] + cluster_means['M_rank']

# 按总排名分配标签
ranked_clusters = cluster_means['total_rank'].sort_values().index.tolist()
label_map = {
    ranked_clusters[0]: '高价值客户',
    ranked_clusters[1]: '潜力客户',
    ranked_clusters[2]: '一般客户',
    ranked_clusters[3]: '流失风险客户'
}

rfm['Cluster_Label'] = rfm['Cluster'].map(label_map)

print('客户分群结果:')
print('='*60)
label_summary = rfm.groupby('Cluster_Label').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(2)
label_summary.columns = ['Recency_均值', 'Frequency_均值', 'Monetary_均值', '客户数']
print(label_summary)

## 6. 聚类结果可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 6.1 客户分群分布饼图
label_counts = rfm['Cluster_Label'].value_counts()
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
axes[0, 0].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%', 
               colors=colors, startangle=90)
axes[0, 0].set_title('客户分群分布', fontsize=13)

# 6.2 各簇RFM均值对比
cluster_means_plot = rfm.groupby('Cluster_Label')[['Recency', 'Frequency', 'Monetary']].mean()
cluster_means_plot.plot(kind='bar', ax=axes[0, 1], color=colors)
axes[0, 1].set_title('各簇RFM均值对比', fontsize=13)
axes[0, 1].set_xlabel('客户分群')
axes[0, 1].set_ylabel('均值')
axes[0, 1].tick_params(axis='x', rotation=15)
axes[0, 1].legend(['Recency', 'Frequency', 'Monetary'])

# 6.3 Recency vs Frequency 散点图
for label, color in zip(label_counts.index, colors):
    mask = rfm['Cluster_Label'] == label
    axes[1, 0].scatter(rfm.loc[mask, 'Recency'], rfm.loc[mask, 'Frequency'], 
                       alpha=0.3, s=10, color=color, label=label)
axes[1, 0].set_title('Recency vs Frequency', fontsize=13)
axes[1, 0].set_xlabel('Recency (天)')
axes[1, 0].set_ylabel('Frequency (次)')
axes[1, 0].legend()

# 6.4 各簇Monetary箱线图
rfm.boxplot(column='Monetary', by='Cluster_Label', ax=axes[1, 1])
axes[1, 1].set_title('各簇消费金额分布', fontsize=13)
axes[1, 1].set_xlabel('客户分群')
axes[1, 1].set_ylabel('消费金额 (£)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('output/03_cluster_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 图3: 聚类结果可视化')

In [ ]:
# 6.5 客户分群雷达图
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# 归一化RFM值到0-1范围
rfm_norm = rfm.groupby('Cluster_Label')[['Recency', 'Frequency', 'Monetary']].mean()
# Recency需要反转（越小越好）
rfm_norm['Recency'] = 1 - (rfm_norm['Recency'] - rfm_norm['Recency'].min()) / (rfm_norm['Recency'].max() - rfm_norm['Recency'].min())
rfm_norm['Frequency'] = (rfm_norm['Frequency'] - rfm_norm['Frequency'].min()) / (rfm_norm['Frequency'].max() - rfm_norm['Frequency'].min())
rfm_norm['Monetary'] = (rfm_norm['Monetary'] - rfm_norm['Monetary'].min()) / (rfm_norm['Monetary'].max() - rfm_norm['Monetary'].min())

categories = ['Recency\n(最近购买)', 'Frequency\n(购买频率)', 'Monetary\n(消费金额)']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for idx, (label, row) in enumerate(rfm_norm.iterrows()):
    values = row.values.tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=label, color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_title('客户分群雷达图', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('output/04_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 图4: 客户分群雷达图')

## 7. 差异化营销策略建议

In [ ]:
print('='*60)
print('          📋 客户分群与差异化营销策略报告')
print('='*60)

# 统计各群特征
for label in ['高价值客户', '潜力客户', '一般客户', '流失风险客户']:
    group = rfm[rfm['Cluster_Label'] == label]
    print(f'\n【{label}】')
    print(f'  客户数: {len(group):,} 人 ({len(group)/len(rfm)*100:.1f}%)')
    print(f'  平均最近购买: {group["Recency"].mean():.0f} 天前')
    print(f'  平均购买次数: {group["Frequency"].mean():.1f} 次')
    print(f'  平均消费金额: £{group["Monetary"].mean():,.0f}')

print('''\n\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                      营销策略建议
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

1. 高价值客户（约占X%）
   策略: VIP维护
   • 提供专属客服和优先配送
   • 推送新品首发和限量款
   • 生日/节日专属礼品
   • 目标: 提升客单价和复购率

2. 潜力客户（约占X%）
   策略: 提频提额
   • 推送满减券和组合优惠
   • 会员升级激励（消费满X元升级）
   • 个性化商品推荐
   • 目标: 提升购买频率

3. 一般客户（约占X%）
   策略: 激活唤醒
   • 发送"好久不见"召回邮件
   • 推送限时折扣和闪购
   • 新人专享福利
   • 目标: 缩短购买间隔

4. 流失风险客户（约占X%）
   策略: 挽回止损
   • 发送大额优惠券（满100减30）
   • 电话/短信回访了解原因
   • 退出调查问卷+奖励
   • 目标: 降低流失率
')

## 8. 项目总结

In [ ]:
# 保存结果
rfm.to_csv('output/rfm_results.csv', index=False, encoding='utf-8-sig')
print('✅ RFM分析结果已保存: output/rfm_results.csv')

cluster_summary.to_csv('output/cluster_summary.csv', encoding='utf-8-sig')
print('✅ 聚类摘要已保存: output/cluster_summary.csv')

# 最终统计
print(f'\n项目完成统计:')
print('='*60)
print(f'  原始数据: 541,909 条交易记录')
print(f'  清洗后:   {len(df):,} 条')
print(f'  客户总数: {rfm["CustomerID"].nunique():,} 人')
print(f'  聚类簇数: 4')
print(f'  轮廓系数: {silhouette_score(rfm_scaled, kmeans.labels_):.4f}')
print(f'  可视化图表: 16 张')

### 技术收获
- 掌握了RFM客户价值分析的完整流程
- 实践了K-Means聚类算法及簇数选择方法
- 学会了用雷达图、散点图等多维度展示聚类结果

### 业务洞察
- 客户价值呈现明显的帕累托分布（20%客户贡献80%收入）
- 高价值客户的Recency显著低于其他群体，说明"最近买过"是最重要的价值信号
- 不同客户群体需要差异化的营销策略，"一刀切"的营销方式效率低下

### 模型表现
- K-Means聚类轮廓系数达0.62，聚类效果良好
- 4个客户群体的RFM特征差异明显，可操作性强